# Statistical testing

Checking two things properly: how good is the simplest possible
forecast (just the sign of one pollster's margin), and how severe is
the multicollinearity notebook 00 flagged.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv('PollingData.csv')
train = df[df.Year.isin([2004, 2008])].copy()

### How good is the 'smart baseline' (just the sign of the Rasmussen poll)?

In [2]:
sub = train.dropna(subset = ['Rasmussen'])
sign_pred = (sub.Rasmussen > 0).astype(int)
acc = (sign_pred == sub.Republican).mean()
print(f'accuracy: {acc:.3f}, n = {len(sub)} (rows where Rasmussen wasn\'t missing)')

pd.crosstab(sub.Republican, sign_pred)

accuracy: 0.948, n = 77 (rows where Rasmussen wasn't missing)


Rasmussen,0,1
Republican,,
0,36,3
1,1,37


A single pollster's sign alone gets 94.8% of states right on the
training years, a genuinely strong baseline for a problem this
one-sided (most states aren't close), and any model built on top of
this has to actually beat that, not just beat a coin flip.

### How severe is the multicollinearity?

In [3]:
X = sm.add_constant(train[['Rasmussen', 'SurveyUSA', 'PropR', 'DiffCount']].dropna())
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    index = X.columns
)
vif

const        27.895251
Rasmussen     6.571293
SurveyUSA     8.159525
PropR        19.200795
DiffCount     8.004411
dtype: float64

In [4]:
X_small = sm.add_constant(train[['SurveyUSA', 'DiffCount']].dropna())
vif_small = pd.Series(
    [variance_inflation_factor(X_small.values, i) for i in range(X_small.shape[1])],
    index = X_small.columns
)
vif_small

const        1.118217
SurveyUSA    1.606590
DiffCount    1.606590
dtype: float64

In [5]:
import json, os
os.makedirs('outputs', exist_ok = True)
with open('outputs/statistical_tests.json', 'w') as f:
    json.dump({
        'smart_baseline_accuracy': float(acc),
        'vif_full': vif.to_dict(),
        'vif_two_variable': vif_small.to_dict(),
    }, f, indent = 2)